In [49]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


COLORS = {
    'specie': {
        "F. Azuke": "#8B0000",             # vinho escuro
        "F. Branco": "#b08968",            # bege claro
        "F. Preto (120 e 520)us": "#000000", # preto
        "F. Roxão (120 e 520)us": "#800080", # roxo escuro
        "F. de Coroa": "#DEB887",          # marrom claro
        "F. Bola": "#A0522D",              # marrom médio
        "F. Rajado": "#CD853F",            # castanho amarelado
        "F. Moyashi": "#B0C4DE",           # azul acinzentado (como feijão moyashi)
        
        "Amassados": "#FFA07A",            # salmão claro
        "Ardidos": "#DC143C",              # vermelho escuro
        "Chochos": "#FFE4B5",              # amarelo claro
        "Danificados": "#8B4513",          # marrom escuro
        "Enrugados": "#D2B48C",            # cor de trigo
        "Esverdeados": "#9ACD32",          # verde limão
        "Fermentados": "#BC8F8F",          # rosa queimado
        "Germinados": "#ADFF2F",           # verde vivo
        "Imaturos": "#EEE8AA",             # amarelo pálido
        "Mancha Café": "#A0522D",          # marrom médio
        "Mancha Púrpura": "#9370DB",       # roxo médio
        "Mofados": "#708090",              # cinza azulado
        "Picados": "#DAA520",              # dourado
        "Quebrados e Partidos": "#CD5C5C", # vermelho queimado
        "Queimados": "#2F4F4F",            # cinza escuro
        "Saudáveis": "#228B22",            # verde floresta
        "Tegumeno Escuro": "#556B2F",      # oliva escuro
    }
}
    


SOJA_CLASSES = [
    "Amassados",
    "Ardidos",
    "Chochos",
    "Danificados",
    "Enrugados",
    "Esverdeados",
    "Fermentados",
    "Germinados",
    "Imaturos",
    "Mancha Café",
    "Mancha Púrpura",
    "Mofados",
    "Picados",
    "Quebrados e Partidos",
    "Queimados",
    "Saudáveis",
    "Tegumeno Escuro",
]

DATA_PATH = "data/"
df = pd.read_csv(f"{DATA_PATH}espectro_graos.csv", index_col=[0])

meta_cols = ['timestamp', 'integration', 'specie']
wave_cols = [c for c in df.columns if c not in meta_cols]

df['spectrum'] = df[wave_cols].values.tolist()
df.drop(columns=wave_cols, inplace=True)

In [50]:
def normalize_sample(spectrum_sample):
    """Realiza a normalização Min-Max do espectro.
    Args:
        spectrum_sample (pd.Series): Espectro bruto da amostra.
    Returns:
        pd.Series: Espectro normalizado com valores entre 0 e 1.
    """
    # spectrum_sample = spectrum_sample.astype(float)
    result  = []
    for value in spectrum_sample:
        norm_spectrum = (value - min(spectrum_sample)) / (max(spectrum_sample) - min(spectrum_sample))
        result.append(norm_spectrum)
    return result

df['spectrum'] = df['spectrum'].apply(normalize_sample)


feijao_mask = df["specie"].str.startswith("F.")
df_feijoes = df[feijao_mask].reset_index(drop=True)

df_soja = df[df["specie"].isin(SOJA_CLASSES)].reset_index(drop=True)

In [52]:
df['integration'].value_counts()

integration
120000.0    620
30000.0     285
400000.0     61
20000.0      42
520000.0     40
300000.0     23
50000.0      23
220000.0     22
100000.0      2
80000.0       2
40000.0       1
Name: count, dtype: int64

### Feijao

In [46]:
def plot_curve_mean(df,
                    column,
                    description: str):

    mean_spec1, std_spec = {}, {}

    for type, specs in df.groupby(column)['spectrum']:
        stack = np.vstack(specs.values)
        mean_spec1[type] = stack.mean(axis=0)
        # Desvio padrão amostral
        std_spec[type] = stack.std(axis=0, ddof=1)
        
    n_bands = np.arange(288)

    fig = go.Figure()
    for type in mean_spec1:
        y_mean = mean_spec1[type]
        y_std = std_spec[type]
        color = COLORS[column].get(type, '#999999')
        
        def _hex_to_rgb(hex_color):
            hex_color = hex_color.lstrip("#")
            return ("rgba" + str(tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4)))).replace(")", ', 0.9)')
        
        color = _hex_to_rgb(color)
        
        fig.add_trace(go.Scatter(
            x=n_bands, y=y_mean + y_std,
            mode='lines',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip'
        ))

        fig.add_trace(go.Scatter(
            x=n_bands, y=y_mean - y_std,
            fill='tonexty',  # preenche entre esse trace e o anterior
            fillcolor=color.replace("0.9", '0.3'),
            line=dict(width=0),
            mode='lines',
            name=f"{type} ±1σ",
            hoverinfo='skip'
        ))

        # Curva da média
        fig.add_trace(go.Scatter(
            x=n_bands, y=y_mean,
            mode='lines',
            name=f"{type} (mean)",
            line=dict(color=color, width=2),
        ))

    # Layout
    fig.update_layout(
        title=description,
        xaxis_title="Índice de Banda",
        yaxis_title="Intensidade",
        template="plotly_white"
    )

    return fig

In [47]:
fig = plot_curve_mean(df_feijoes,
                      'specie',
                      'Espectro médio dos Feijões')
fig.show()

### Soja

In [51]:
fig = plot_curve_mean(df_soja,
                      'specie',
                      'Espectro médio das sojas')
fig.show()